In [0]:
%pip install aiohttp
%restart_python

In [0]:
import asyncio
import aiohttp
import io
from azure.storage.blob.aio import BlobServiceClient
import json 
import requests
from databricks.sdk import WorkspaceClient
from datetime import datetime

In [0]:
dbutils.widgets.text("my_scope", "def-scope")
my_scope = dbutils.widgets.get("my_scope")

dbutils.widgets.text("catalog_name", "workspace")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
workspace_url = "https://dbc-3b714993-952c.cloud.databricks.com/"
db_token = dbutils.secrets.get(scope=my_scope, key="volume_token")

w = WorkspaceClient(host=workspace_url, token=db_token)
VOLUME_BASE_PATH = f"/Volumes/{catalog_name}/crime_bronze/streaming"

async def run(session, url):
    lines = []    

    async with session.get(url) as response:
        async for line in response.content:
            decoded_line = line.decode('utf-8').strip()
            lines.append(decoded_line)

            if len(lines) >= 50:
                content = "\n".join(lines)
                current_time = datetime.now().strftime("%H-%M-%S")
                blob_name = f"data_{current_time}.jsonl"

                w.files.upload(f"{VOLUME_BASE_PATH}/{blob_name}", io.BytesIO(content.encode("utf-8")), overwrite=True)
                print(f"Saved file {blob_name}, number of lines: {len(lines)}")

                lines = []
                

    if lines:
        content = "\n".join(lines)
        current_time = datetime.now().strftime("%H-%M-%S")
        blob_name = f"data_{current_time}.jsonl"

        w.files.upload(f"{VOLUME_BASE_PATH}/{blob_name}", io.BytesIO(content.encode("utf-8")), overwrite=True)
        print(f"Saved file {blob_name}, number of lines: {len(lines)}")

In [0]:
w = WorkspaceClient()
app_client_id = w.apps.get("csv-api-app").oauth2_app_client_id

In [0]:
url = "https://dbc-3b714993-952c.cloud.databricks.com//oidc/v1/token"

notebook_token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

data = {
    "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
    "subject_token": notebook_token,
    "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
    "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
    "scope": "all-apis",
    "audience": app_client_id,
}

response = requests.post(url=url, data=data)
audience_token = response.json()["access_token"]

In [0]:
headers = {"Authorization": f"Bearer {audience_token}"}
custom_timeout = aiohttp.ClientTimeout(total=None)

In [0]:
URL = "https://csv-api-app-7474645174283015.aws.databricksapps.com/api/stream" 

async with aiohttp.ClientSession(headers=headers, timeout = custom_timeout) as session:
    await run(session, URL) 